In [9]:
from neo4j import GraphDatabase
from CM.utils import *
from CM.email import *
from CM.USES import *
import pandas as pd
import json

In [10]:
def is_valid_json(json_string):
    try:
        json.loads(json_string)
        return True
    except json.JSONDecodeError:
        return False

In [11]:
def validateJSON(database, property = 'parentContext', path ="/mnt/storage/app/tmp/invalid_json.xlsx" ):
    try:

        driver = getDriver(database)

        query = f"match (c:CATEGORY)<-[r:USES]-(d:DATASET) where not r.{property} is null return d.CMID as datasetID, c.CMID as CMID, r.Key as Key, r.{property} as prop"

        results = getQuery(query, driver)

        results = pd.DataFrame(results)

        results = pd.DataFrame.explode(results, "prop")

        results["is_valid_json"] = results["prop"].apply(is_valid_json)

        invalid = results[results["is_valid_json"] == False]

        invalid.to_excel(path)

        invalid = invalid.to_dict(orient="records")

        return(invalid)

    except Exception as e:
        return "Unable to validate JSON properties: " + str(e)
    

In [12]:
def checkDomains(data,database):
    try:
        if data is None:
            data = False
        elif data.lower() == "true":
            data = True
        else:
            data = False
        driver = getDriver(database)
        query = """
        match (n:CATEGORY)
        where size(labels(n)) = 1
        optional match (n)<-[r:USES]-(d:DATASET) 
        return "CATEGORY" as query, n.CMID as CMID, n.CMName as CMName, r.label as label, d.CMID as datasetID
        UNION ALL
        match (n)
        where isEmpty([i in labels(n) where i in ["DATASET","METADATA","USER","CATEGORY","DELETED"]]) 
        optional match (n)<-[r:USES]-(d:DATASET) 
        return "MissingCATEGORY" as query,  n.CMID as CMID, n.CMName as CMName, r.label as label, d.CMID as datasetID
        """
        result = getQuery(query, driver)
        
        if data:
            return result
        else:
            return "Completed"
         
    except Exception as e:
        return str(e)
    


In [13]:
database = "SocioMap"

In [14]:
fp1 = "tmp/invalid_json_geoCoords.xlsx"
results1 = validateJSON(
    database=database, property='geoCoords', path=fp1)


In [17]:
len(results1)

0

In [18]:
fp2 = "tmp/invalid_json_parentContext.xlsx"
results2 = validateJSON(
    database=database, property='parentContext', path=fp2)

In [19]:
len(results2)

0